<a href="https://colab.research.google.com/github/Lahariravikanti3121/Real-time-Health-Monitoring-System/blob/main/Pred_target_using_diag.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import torch
import pandas as pd
import os
from PIL import Image
from tqdm import tqdm
from transformers import LlavaProcessor, LlavaForConditionalGeneration
from sklearn.metrics import accuracy_score, f1_score
import torch.nn.functional as F

In [ ]:
model_id = "llava-hf/llava-1.5-7b-hf"

processor = LlavaProcessor.from_pretrained(model_id)
model = LlavaForConditionalGeneration.from_pretrained(
    model_id,
    torch_dtype=torch.float16,
    device_map="auto"
)

model.eval()

In [ ]:
import pandas as pd
import torch
from sklearn.metrics import accuracy_score, classification_report

# --- 1. Load Data ---
metadata_path = '/content/drive/MyDrive/metadata.csv'
df = pd.read_csv(metadata_path)

# Standardize to avoid string matching errors
df["diagnosis"] = df["diagnosis"].str.lower().str.strip()
df["benign_malignant"] = df["benign_malignant"].str.lower().str.strip()

# --- 2. Configuration & Logic ---
# We use a strict prompt to force a one-word answer
PROMPT_TEMPLATE = "A patient has a skin lesion diagnosed as '{diagnosis}'. Is this lesion malignant or benign? Answer with only one word: 'malignant' or 'benign'.\nAnswer:"

def extract_label(text):
    text = text.lower().strip()
    if "malignant" in text: return "malignant"
    if "benign" in text: return "benign"
    return "unknown"

# --- 3. Run Inference ---
df_sample = df.copy()
predictions = []

print(f"Starting experiment on {len(df_sample)} samples using LLaVA...")

for diagnosis in df_sample["diagnosis"]:
    prompt = PROMPT_TEMPLATE.format(diagnosis=diagnosis)

    # LLaVA processor handles the text/image formatting
    inputs = processor(text=prompt, return_tensors="pt").to(model.device)

    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=10,
            do_sample=False,  # Greedy search for deterministic 'lookup'
        )

    # CRITICAL: Decode only the newly generated tokens
    # LLaVA includes the prompt in the output, so we slice it
    new_tokens = output_ids[0][inputs.input_ids.shape[-1]:]
    response = processor.decode(new_tokens, skip_special_tokens=True)

    pred = extract_label(response)
    predictions.append(pred)

# --- 4. Store and Compare ---
df_sample["prediction"] = predictions

# --- 5. Analysis ---
accuracy = accuracy_score(df_sample["benign_malignant"], df_sample["prediction"])

print("\n" + "="*30)
print(f"EXPERIMENT RESULTS (Diagnosis -> Status)")
print("="*30)
print(f"Overall Accuracy: {accuracy:.2%}")
print("\nFirst 10 Rows Comparison:")
print(df_sample[["diagnosis", "benign_malignant", "prediction"]])

# Check for mismatches (this is where the leakage is confirmed)
# mismatches = df_sample[df_sample["benign_malignant"] != df_sample["prediction"]]
# if mismatches.empty:
#     print("\nResult: 100% Match. This confirms the model is using the diagnosis string as a direct label lookup.")
# else:
#     print(f"\nResult: {len(mismatches)} mismatches found. This may be due to 'unknown' extractions or obscure medical terms.")